# Dados hidrometeorológicos da ANA

Leitura dos arquivos XML da estação 87444000 (Usina do Gasômetro, Porto Alegre).

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd

PROJECT_DIR = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
DATA_DIR = PROJECT_DIR / "dados" / "originais" / "ana"

def read_ana_xml(path: Path) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for record in root.findall(".//DadosHidrometereologicos"):
        rows.append({
            child.tag: child.text
            for child in record
        })

    return pd.DataFrame(rows)


xml_files = sorted(DATA_DIR.glob("ana_*.xml"))
df_ana = pd.concat(
    [read_ana_xml(path) for path in xml_files],
    ignore_index=True,
)
df_ana["DataHora"] = pd.to_datetime(df_ana["DataHora"].str.strip())
for column in ["CodEstacao", "Vazao", "Nivel", "Chuva"]:
    df_ana[column] = pd.to_numeric(df_ana[column], errors="coerce")

df_ana = df_ana.sort_values("DataHora").reset_index(drop=True)
df_ana

: 

In [ ]:
df_ana.info()
df_ana.describe(include="all")